# Cylinder VTK -> MeshGraphNet dataset

OpenFOAM が出力した `internal.vtu` の時系列を読み込み、2D 平面グラフへ変換して `.pt` に保存します。

このNotebookでは、実用スクリプトの `parse_args()` や汎用的なCLI処理を省き、次の順序だけを追います。

1. VTKを読む
2. 平面メッシュの座標とエッジを作る
3. `U=(u,v)` と `p` をノード特徴量にする
4. 次時刻を教師信号にした `torch_geometric.data.Data` を保存する

## 1. 必要ライブラリと最小設定

`pyvista` でVTKを読み、NumPyでメッシュを扱い、PyTorch Geometricの `Data` に詰めます。

In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv
import torch

DTYPE = torch.float32
torch.manual_seed(0)

print(torch.__version__, pv.__version__)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.12.0a0+0291f960b6.nv26.04.48445190 0.48.4


## 2. 入力・出力パス

`parse_args()` の代わりに、実験に合わせてこのセルを書き換えます。ここでは1つのケースの `internal.vtu` 時系列を対象にします。

In [8]:
input_dir = Path("data/raw")
output_dir = Path("data/preprocessed")
case_dirs = ["VTK_000", "VTK_001", "VTK_002"]
target_fields = ("U", "p")
boundary_names = (("wall", 1), ("inlet", 2), ("outlet", 3))

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_vtk(path: Path) -> pv.UnstructuredGrid:
    """VTKファイルを読み込む関数"""
    mesh = pv.read(path)
    if not isinstance(mesh, pv.UnstructuredGrid):
        mesh = mesh.cast_to_unstructured_grid()
    return mesh


def time_key(path: Path) -> int:
    """各VTKファイルがどの時間ステップのものか取得する関数"""
    return int(path.parent.name.rsplit("_", maxsplit=1)[1])

def snapshot_files(case_dir: Path) -> list[Path]:
    """ケースディレクトリ内の全internal.vtuファイルを取得する関数"""
    files = sorted(case_dir.glob("*/internal.vtu"), key=time_key)
    if not files:
        raise FileNotFoundError(f"internal.vtu がありません: {case_dir}")
    return files

元のメッシュは厚み方向(z方向)に1セル分押し出されているため、小さい方のz平面だけを2Dノードとして使います。

In [ ]:
def boundary_node_types(snapshot_dir: Path, pos: np.ndarray) -> np.ndarray:
    """境界上の各ノードがどの境界に属する関数か判定する関数"""
    node_type = np.zeros(len(pos), dtype=np.int64)
    lookup = {tuple(np.round(point, 7)): index for index, point in enumerate(pos)}
    for name, value in boundary_names:
        boundary = pv.read(snapshot_dir / "boundary" / f"{name}.vtp")
        for point in boundary.points[:, :2]:
            index = lookup.get(tuple(np.round(point, 7)))
            if index is not None:
                node_type[index] = value
    return node_type


def extract_snapshot(mesh: pv.UnstructuredGrid) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """小さい方のz平面のノードの座標、エッジ接続、物理量を抽出する関数"""
    z = mesh.points[:, 2]
    plane_ids = np.flatnonzero(np.isclose(z, z.min()))
    pos = np.asarray(mesh.points[plane_ids, :2], dtype=np.float32)
    source_to_node = np.full(mesh.n_points, -1, dtype=np.int64)
    source_to_node[plane_ids] = np.arange(len(plane_ids))

    edges = set()
    for cell_index in range(mesh.n_cells):
        ids = source_to_node[np.asarray(mesh.get_cell(cell_index).point_ids)]
        ids = np.unique(ids[ids >= 0])
        if len(ids) < 3:
            continue
        center = pos[ids].mean(axis=0)
        order = ids[np.argsort(np.arctan2(pos[ids, 1] - center[1], pos[ids, 0] - center[0]))]
        edges.update((min(a, b), max(a, b)) for a, b in zip(order, np.roll(order, -1)))

    undirected = np.asarray(sorted(edges), dtype=np.int64)
    edge_index = np.concatenate((undirected, undirected[:, ::-1]), axis=0).T
    velocity = np.asarray(mesh.point_data[target_fields[0]])[plane_ids, :2]
    pressure = np.asarray(mesh.point_data[target_fields[1]])[plane_ids].reshape(-1, 1)
    state = np.concatenate((velocity, pressure), axis=1).astype(np.float32)
    return pos, edge_index, state

3ケースの教師データを作成します。ケースごとに `pos`, `edge_index`, `node_type`, `state`, `case_id` を抽出し、ひとつのptファイルに出力します。

In [ ]:
def preprocess_case(case_id: str, case_dir: Path) -> dict[str, object]:
    files = snapshot_files(case_dir)
    first_mesh = read_vtk(files[0])
    pos, edge_index, first_state = extract_snapshot(first_mesh)
    node_type = boundary_node_types(files[0].parent, pos)
    states = [first_state]

    for path in files[1:]:
        current_pos, current_edges, current_state = extract_snapshot(read_vtk(path))
        if not np.allclose(current_pos, pos) or not np.array_equal(current_edges, edge_index):
            raise ValueError(f"メッシュが変化しています: {path}")
        states.append(current_state)

    return {
        "case_id": case_id,
        "pos": torch.tensor(pos, dtype=DTYPE),
        "edge_index": torch.tensor(edge_index, dtype=torch.long),
        "node_type": torch.tensor(node_type, dtype=torch.long),
        "state": torch.tensor(np.stack(states), dtype=DTYPE),
    }

processed_cases = []
for index, case_dir in enumerate(case_dirs):
    case_id = f"{index:03d}"
    case = preprocess_case(case_id, case_dir)
    output_path = output_dir / f"case_{case_id}.pt"
    torch.save(case, output_path)
    processed_cases.append(case)
    print(f"{output_path}: {case['state'].shape[0]} snapshots, {case['state'].shape[1]} nodes")